# Lecture 2b — Generics, classes, and the `typing` module

The second half of Lecture 2. **[Lecture 2a](Lecture2a.ipynb)** showed how to *write*
annotations; this half is about how the type checker *reasons* with them — which is what
you need as soon as your code becomes generic or object-oriented.

It also settles the two questions 2a left open: how to type a function that works on
lists of anything (`TypeVar`), and how to type an argument that may be `None`
(`Optional`).


## Objectives
By the end of this half you can:
- reason about **subtypes**, and about **covariance / contravariance / invariance**;
- explain **gradual typing** and tell **consistency** from **subtyping**;
- write generic code with **`TypeVar`** (plain, constrained, and `bound=`);
- use `Optional` / `Union` for arguments that may be `None`;
- annotate methods, classes-as-types, forward references, `self` / `cls`,
  `*args` / `**kwargs`, and `Callable`;
- find what you need in the **`typing`** module, and follow **PEP 8**.


## Where we left off

From **Lecture 2a**: Python is dynamically typed and stays that way — annotations have
**no runtime effect**; a static checker such as `mypy` is what reads them. You can
annotate parameters, return values (`-> None` included) and composite data
(`list[str]`, `tuple[str, str]`, `dict[str, bool]`), and hide an ugly shape behind an
alias. `Any` is the escape hatch that silences the checker.

Two annotations defeated us:

```python
def choose(items: Sequence[Any]) -> Any:       # works on names AND cards -- but the
    return random.choice(items)                # checker now knows nothing about either

def player_order(names, start=None):           # `start` is a str ... or None
    ...
```

Both are fixed in this notebook, after one detour through theory. The detour is short
and it is the reason the fixes make sense.

## How to read this lecture

| Section | Read | What it gives you |
|---|---|---|
| 1 · Type theory | 7 min | subtype, variance, consistency |
| 2 · Type variables and `Optional` | 6 min | the answer to both open questions |
| 3 · Classes, `self`/`cls`, `Callable` | 10 min | typing object-oriented code |
| 4 · The `typing` module | *skim* | a reference to come back to, not a read |
| **End to end** | **≈ 28 min** | enough for Labwork 2, exercises 3–4 |

Section 4 is deliberately a catalogue. Skim the headings once so you know what exists,
then come back to it when `mypy` asks you for a type you have not met.


## Type theory
This lecture does not cover all the theory underpinning Python type hints. For more details see the PEP 483 and 484. 

### Subtypes
One important concept is that of subtypes. Formally, we say that a type `T` is a subtype of `U` if the following two conditions hold:
- Every value from `T` is also in the set of values of `U` type.
- Every function from `U` type is also in the set of functions of `T` type.

These two conditions guarantees that even if type `T` is different from `U`, variables of type `T` can always pretend to be `U`.

For a concrete example, consider `T = bool` and `U = int`. The `bool` type takes only two values. Usually these are denoted `True` and `False`, but these names are just aliases for the integer values `1` and `0`, respectively:
```python
>>> int(False)
0
>>> int(True)
1
>>> True + True
2
>>> issubclass(bool, int)
True
```
Since `0` and `1` are both integers, the first condition holds. Above you can see that booleans can be added together, but they can also do anything else integers can. This is the second condition above. In other words, `bool` is a subtype of `int`.

The importance of subtypes is that a subtype can always pretend to be its supertype. For instance, the following code type checks as correct:
```python
def double(number: int) -> int:
    return number * 2

print(double(True))  # Passing in bool instead of int
```
Subtypes are somewhat related to subclasses. In fact all subclasses corresponds to subtypes, and `bool` is a subtype of `int` because `bool` is a subclass of `int`. However, there are also subtypes that do not correspond to subclasses. For instance `int` is a subtype of `float`, but `int` is not a subclass of `float`.

In [ ]:
# `bool` is a subtype of `int` -- both conditions, checked at run time.
print(int(False), int(True))        # 1. every bool value IS an int value
print(True + True)                  # 2. every int operation works on a bool
print(issubclass(bool, int))        # and here it is also a subCLASS

def double(number: int) -> int:
    return number * 2

print(double(True))                 # a subtype can always pretend to be its supertype

# Careful: subtype and subclass are not the same thing.
print(issubclass(int, float))       # False -- yet `int` IS a subtype of `float`
def half(x: float) -> float:
    return x / 2
print(half(7))                      # mypy accepts this: int is consistent with float


### Covariant, Contravariant, and Invariant
What happens when you use subtypes inside composite types? For instance, is `Tuple[bool]` a subtype of `Tuple[int]`? The answer depends on the composite type, and whether that type is **covariant**, **contravariant**, or **invariant**. This gets technical fast, so let’s just give a few examples:
- `Tuple` is *covariant*. This means that *it preserves the type hierarchy of its item types*: `Tuple[bool]` is a subtype of `Tuple[int]` because `bool` is a subtype of `int`.
- `List` is *invariant*. *Invariant types give no guarantee about subtypes*. While all values of `List[bool]` are values of `List[int]`, you can append an `int` to `List[int]` and not to `List[bool]`. In other words, the second condition for subtypes does not hold, and `List[bool]` is not a subtype of `List[int]`.
- `Callable` is *contravariant* in its arguments. This means that *it reverses the type hierarchy*. You will see how `Callable` works later, but for now think of `Callable[[T], ...]` as a function with its only argument being of type `T`. An example of a `Callable[[int], ...]` is the `double()` function defined above. Being contravariant means that if a function operating on a `bool` is expected, then a function operating on an `int` would be acceptable.

In general, you don’t need to keep these expression straight. However, you should be aware that subtypes and composite types may not be simple and intuitive.

> ### Aide-mémoire — which way does the arrow go?
>
> Variance answers one question: *if `T` is a subtype of `U`, is `C[T]` a subtype of
> `C[U]`?* The rule of thumb is **what the container lets you do with it**:
>
> | You can only... | Variance | Example | `C[bool]` is a subtype of `C[int]`? |
> |---|---|---|---|
> | **read** from it | covariant | `Tuple`, `Sequence`, `Iterable`, `FrozenSet` | **yes** |
> | **read and write** | invariant | `List`, `Dict`, `Set` | **no** |
> | **feed** values into it | contravariant | `Callable` *arguments* | **reversed** |
>
> The invariance of `List` is the one that bites in practice, and the reason is exactly
> the second condition for subtypes: you can `append(42)` to a `List[int]`, and you must
> not be able to do that to a `List[bool]` — so `List[bool]` cannot pretend to be a
> `List[int]`.
>
> **The practical consequence, worth remembering for Labwork 2 and for Week 2:** annotate
> a *parameter* with the most permissive thing that works — `Sequence[float]`, not
> `list[float]` — and annotate a *return value* concretely — `-> list[float]`. The next
> cells measure the difference.


In [ ]:
import os, sys

# The interpreter that is running this notebook.
# On macOS there is often no `python` command at all, only `python3`, and adding a
# shell alias does not help: `!` cells run a NON-interactive shell that never reads
# your profile, and `%%script` launches the program directly, without any shell.
# `sys.executable` is an absolute path, so it keeps working after a `cd` too.
PY = sys.executable
os.makedirs("code", exist_ok=True)
print("code/ ready")
print(f'interpreter: {PY}')


In [ ]:
%%writefile code/variance_demo.py
from collections.abc import Sequence


def total_seq(xs: Sequence[float]) -> float:   # covariant parameter
    return sum(xs)


def total_list(xs: list[float]) -> float:      # invariant parameter
    return sum(xs)


ints: list[int] = [1, 2, 3]
bools: list[bool] = [True, False]
pair: tuple[float, ...] = (1.0, 2.0)

total_seq(ints)       # 1
total_seq(bools)      # 2
total_seq(pair)       # 3
total_seq([1, 2])     # 4

total_list(ints)      # 5
total_list(bools)     # 6
total_list(pair)      # 7
total_list([1, 2])    # 8


In [ ]:
!{PY} -m mypy --strict code/variance_demo.py


Calls 1–4 all pass: `Sequence[float]` is covariant, so it swallows a `list[int]`, a
`list[bool]` and a `tuple[float, ...]` without complaint.

Calls 5, 6 and 7 are errors — `list` is invariant, and `mypy` even volunteers the fix:

```
error: Argument 1 to "total_list" has incompatible type "list[int]"; expected "list[float]"
note: "list" is invariant -- see .../common_issues.html#variance
note: Consider using "Sequence" instead, which is covariant
```

**Call 8 passes, and that is the trap.** `[1, 2]` is a *literal*, so `mypy` infers its
element type from the parameter it is being passed to and happily makes it a
`list[float]`. So a badly annotated function looks perfectly fine while you test it with
literals, and only breaks the day a caller passes a variable it already had. Do not read
"my test call type-checked" as "my annotation is right".


### Gradual Typing and Consistent Types
Earlier we mentioned that Python supports gradual typing, where you can gradually add type hints to your Python code. Gradual typing is essentially made possible by the `Any` type.

Somehow `Any` sits both at the top and at the bottom of the type hierarchy of subtypes. `Any` type behaves as if it is a subtype of `Any`, and `Any` behaves as if it is a subtype of any other type. Looking at the definition of subtypes above this is not really possible. Instead we talk about consistent types.

The type `T` is consistent with the type `U` if `T` is a subtype of `U` or either `T` or `U` is `Any`.

The type checker only complains about inconsistent types. The takeaway is therefore that you will never see type errors arising from the `Any` type.

This means that you can use `Any` to explicitly fall back to dynamic typing, describe types that are too complex to describe in the Python type system, or describe items in composite types. For instance, a dictionary with `string` keys that can take any type as its values can be annotated `Dict[str, Any]`.

Do remember, though, if you use `Any` the static type checker will effectively not do any type any checking.

## Playing with types - 2
Let’s return to our practical examples. Recall that you were trying to annotate the general choose() function:
```python
import random
from typing import Any, Sequence

def choose(items: Sequence[Any]) -> Any:
    return random.choice(items)
```

### Type Variables
A type variable is a special variable that can take on any type, depending on the situation.

Let’s create a type variable that will effectively encapsulate the behavior of `choose()`:
```python
import random
from typing import Sequence, TypeVar

Choosable = TypeVar("Choosable")

def choose(items: Sequence[Choosable]) -> Choosable:
    return random.choice(items)

names = ["Guido", "Jukka", "Ivan"]
reveal_type(names)

name = choose(names)
reveal_type(name)
```
A type variable must be defined using `TypeVar` from the typing module. When used, a type variable ranges over all possible types and takes the most specific type possible. In the example, name is now a `str`:
```bash
$ mypy choose.py
choose.py:10: note: Revealed type is "builtins.list[builtins.str]"
choose.py:12: note: Revealed type is "builtins.str"
```
Consider a few other examples:
```python
# choose_examples.py
from choose import choose

reveal_type(choose(["Guido", "Jukka", "Ivan"]))
reveal_type(choose([1, 2, 3]))
reveal_type(choose([True, 42, 3.14]))
reveal_type(choose(["Python", 3, 7]))
```
The first two examples should have type `str` and `int`, but what about the last two? 
The individual list items have different types, and in that case the `Choosable` type variable does its best to accommodate:
```bash
$ mypy choose_examples.py
choose_examples.py:4: note: Revealed type is "builtins.str"
choose_examples.py:5: note: Revealed type is "builtins.int"
choose_examples.py:6: note: Revealed type is "builtins.float"
choose_examples.py:7: note: Revealed type is "builtins.object"
```
As you’ve already seen `bool` is a subtype of `int`, which again is a subtype of `float`. 
So in the third example the return value of `choose()` is guaranteed to be something that can be thought of as a `float`. 
In the last example, there is no subtype relationship between `str` and `int`, so the best that can be said about the return value is that it is an `object`.

Note that none of these examples raised a type error. 
Is there a way to tell the type checker that `choose()` should accept both strings and numbers, but not both at the same time?

You can constrain type variables by listing the acceptable types thanks to the `TypeVar` typing:
```python
# choose.py
import random

from typing import Sequence, TypeVar

Choosable = TypeVar("Choosable", str, float)

def choose(items: Sequence[Choosable]) -> Choosable:
    return random.choice(items)

reveal_type(choose(["Guido", "Jukka", "Ivan"]))
reveal_type(choose([1, 2, 3]))
reveal_type(choose([True, 42, 3.14]))
reveal_type(choose(["Python", 3, 7]))
```
Now `Choosable` can only be either `str` or `float`, and `mypy` will note that the last example is an error:
```bash
$ mypy choose.py
choose.py:10: error: Revealed type is 'builtins.str*'
choose.py:11: error: Revealed type is 'builtins.float*'
choose.py:12: error: Revealed type is 'builtins.float*'
choose.py:13: error: Revealed type is 'builtins.object*'
choose.py:13: error: Value of type variable "Choosable" of "choose" cannot be "object"
```
Also note that in the second example the type is considered `float` even though the input list only contains `int` objects. 
This is because `Choosable` was restricted to strings and floats and `int` is a subtype of `float`.

In our card game we want to restrict `choose()` to be used for `str` and `Card`:
```python
Choosable = TypeVar("Choosable", str, Card)

def choose(items: Sequence[Choosable]) -> Choosable:
    ...
```
We briefly mentioned that `Sequence` represents both lists and tuples. 
As we noted, a `Sequence` can be thought of as a duck type, since it can be any object with `.__len__()` and `.__getitem__()` implemented.

> ### Three flavours of `TypeVar`, and when to use which
>
> ```python
> T         = TypeVar("T")                        # 1. unconstrained: any type at all
> Choosable = TypeVar("Choosable", str, Card)     # 2. constrained: EXACTLY str or Card
> TAnimal   = TypeVar("TAnimal", bound="Animal")  # 3. bounded: Animal or any subclass
> ```
>
> 1. **Unconstrained** — for a function that does not care, only relays: `choose`, a
>    generic container, `def first(xs: Sequence[T]) -> T`.
> 2. **Constrained** — a closed list of alternatives. The variable takes one of the listed
>    types, never a common supertype; `choose(["Python", 3, 7])` is an *error* rather than
>    silently widening to `object`.
> 3. **Bounded** — "this type, or anything below it". The form you need for `self` and
>    `cls`, in the `Animal`/`Dog` example two sections below.
>
> Flavour 1 is what **Labwork 2, exercise 3** needs for the linked list; flavour 3 is what
> makes `Dog.newborn()` return a `Dog` instead of an `Animal`.


### Optional Type
A common pattern in Python is to use `None` as a default value for an argument. This is usually done either to avoid problems with `mutable` default values or to have a sentinel value flagging special behavior.

In the card example, the `player_order()` function uses `None` as a sentinel value for start saying that if no start player is given it should be chosen randomly:
```python
def player_order(names, start = None):
    """Rotate player order so that start goes first"""
    if start is None:
        start = choose(names)
    start_idx = names.index(start)
    return names[start_idx:] + names[:start_idx]
```
The challenge this creates for type hinting is that in general start should be a string. 
However, it may also take the special non-string value `None`.

In order to annotate such arguments you can use the `Optional` type:
```python
from typing import Sequence, Optional

def player_order(
    names: Sequence[str], start: Optional[str] = None
) -> Sequence[str]:
    ...
```
The `Optional` type simply says that a variable either has the type specified or is `None`. An equivalent way of specifying the same would be using the `Union` type: `Union[None, str]`.

### The Object(ive) of the Game
Let’s rewrite the card game to be more object-oriented. 
This will allow us to discuss how to properly annotate classes and methods.

A more or less direct translation of our card game into code that uses classes for `Card`, `Deck`, `Player`, and `Game` looks something like the following:

In [ ]:
%%script {PY}
import random
import sys

class Card:
    SUITS = "♠ ♡ ♢ ♣".split()
    RANKS = "2 3 4 5 6 7 8 9 10 J Q K A".split()

    def __init__(self, suit, rank):
        self.suit = suit
        self.rank = rank

    def __repr__(self):
        return f"{self.suit}{self.rank}"

class Deck:
    def __init__(self, cards):
        self.cards = cards

    @classmethod
    def create(cls, shuffle=False):
        """Create a new deck of 52 cards"""
        cards = [Card(s, r) for r in Card.RANKS for s in Card.SUITS]
        if shuffle:
            random.shuffle(cards)
        return cls(cards)

    def deal(self, num_hands):
        """Deal the cards in the deck into a number of hands"""
        cls = self.__class__
        return tuple(cls(self.cards[i::num_hands]) for i in range(num_hands))

class Player:
    def __init__(self, name, hand):
        self.name = name
        self.hand = hand

    def play_card(self):
        """Play a card from the player's hand"""
        card = random.choice(self.hand.cards)
        self.hand.cards.remove(card)
        print(f"{self.name}: {card!r:<3}  ", end="")
        return card

class Game:
    def __init__(self, *names):
        """Set up the deck and deal cards to 4 players"""
        deck = Deck.create(shuffle=True)
        self.names = (list(names) + "P1 P2 P3 P4".split())[:4]
        self.hands = {
            n: Player(n, h) for n, h in zip(self.names, deck.deal(4))
        }

    def play(self):
        """Play a card game"""
        start_player = random.choice(self.names)
        turn_order = self.player_order(start=start_player)

        # Play cards from each player's hand until empty
        while self.hands[start_player].hand.cards:
            for name in turn_order:
                self.hands[name].play_card()
            print()

    def player_order(self, start=None):
        """Rotate player order so that start goes first"""
        if start is None:
            start = random.choice(self.names)
        start_idx = self.names.index(start)
        return self.names[start_idx:] + self.names[:start_idx]

if __name__ == "__main__":
    # Read player names from command line
    player_names = sys.argv[1:]
    game = Game(*player_names)
    game.play()

### Type Hints for Methods
Now let’s add types to this code.

First of all type hints for methods work much the same as type hints for functions. The only difference is that the `self` argument need not be annotated, as it always will be a class instance. The types of the `Card` class are easy to add:
```python
class Card:
    SUITS = "♠ ♡ ♢ ♣".split()
    RANKS = "2 3 4 5 6 7 8 9 10 J Q K A".split()

    def __init__(self, suit: str, rank: str) -> None:
        self.suit = suit
        self.rank = rank

    def __repr__(self) -> str:
        return f"{self.suit}{self.rank}"
```
Note that the `.__init__()` method always should have `None` as its return type.

### Classes as Types
There is a correspondence between classes and types. For example, all instances of the `Card` class together form the `Card` type. To use classes as types you simply use the name of the class.

For example, a `Deck` essentially consists of a list of `Card` objects. You can annotate this as follows:
```python
class Deck:
    def __init__(self, cards: List[Card]) -> None:
        self.cards = cards
```
`Mypy` is able to connect your use of `Card` in the annotation with the definition of the `Card` class.

This doesn’t work as cleanly though when you need to refer to the class currently being defined. For example, the `Deck.create()` class method returns an object with type `Deck`. However, you can’t simply add `-> Deck` as the `Deck` class is **not yet fully defined**.

Instead, you are allowed to use string literals in annotations. These strings will only be evaluated by the type checker later, and can therefore **contain self and forward references**. The `.create()` method should use such string literals for its types:
```python
class Deck:
    @classmethod
    def create(cls, shuffle: bool = False) -> "Deck":
        """Create a new deck of 52 cards"""
        cards = [Card(s, r) for r in Card.RANKS for s in Card.SUITS]
        if shuffle:
            random.shuffle(cards)
        return cls(cards)
```
Note that the `Player` class also will reference the `Deck` class. This is however no problem, since `Deck` is defined before `Player`:
```python
class Player:
    def __init__(self, name: str, hand: Deck) -> None:
        self.name = name
        self.hand = hand
```
Usually annotations are not used at runtime. This has given wings to the idea of postponing the evaluation of annotations. Instead of evaluating annotations as Python expressions and storing their value, the proposal is to store the string representation of the annotation and only evaluate it when needed.

Automatic postponed evaluation of annotations was proposed in [PEP 563](https://peps.python.org/pep-0563/) to become the default (originally targeted at Python 3.10), but it was deferred; a revised mechanism ([PEP 649](https://peps.python.org/pep-0649/)) is planned for a later version. In the meantime, since Python 3.7, forward references are available through a `__future__` import:
```python
from __future__ import annotations

class Deck:
    @classmethod
    def create(cls, shuffle: bool = False) -> Deck:
        ...
```
With the `__future__` import you can use `Deck` instead of `"Deck"` even before `Deck` is defined.

### Returning `self` or `cls`
As noted, you should typically not annotate the `self` or `cls` arguments. Partly, this is not necessary as `self` points to an instance of the class, so it will have the type of the class. In the `Card` example, `self` has the implicit type `Card`. Also, adding this type explicitly would be cumbersome since the class is not defined yet. You would have to use the string literal syntax, self: `"Card"`.

There is one case where you might want to annotate `self` or `cls`, though. Consider what happens if you have a superclass that other classes inherit from, and which has methods that return `self` or `cls`:
```python
# dogs.py
from datetime import date

class Animal:
    def __init__(self, name: str, birthday: date) -> None:
        self.name = name
        self.birthday = birthday

    @classmethod
    def newborn(cls, name : str) -> 'Animal':
        return cls(name, date.today())

    def twin(self, name: str) -> "Animal":
        cls = self.__class__
        return cls(name, self.birthday)

class Dog(Animal):
    def bark(self) -> None:
        print(f"{self.name} says woof!")

fido = Dog.newborn("Fido")
pluto = fido.twin("Pluto")
fido.bark()
pluto.bark()
```
While the code runs without problems, `mypy` will flag a problem:
```bash
$ mypy dogs.py
dogs.py:22: error: "Animal" has no attribute "bark"
dogs.py:23: error: "Animal" has no attribute "bark"
```
The issue is that even though the inherited `Dog.newborn()` and `Dog.twin()` methods will return a `Dog` the annotation says that they return an `Animal`.

In cases like this you want to be more careful to make sure the annotation is correct. The return type should match the type of `self` or the instance type of `cls`. This can be done using **type variables** that keep track of what is actually passed to `self` and `cls`:
```python
# dogs.py
from datetime import date
from typing import Type, TypeVar

TAnimal = TypeVar("TAnimal", bound="Animal")

class Animal:
    def __init__(self, name: str, birthday: date) -> None:
        self.name = name
        self.birthday = birthday

    @classmethod
    def newborn(cls: Type[TAnimal], name: str) -> TAnimal:
        return cls(name, date.today())

    def twin(self: TAnimal, name: str) -> TAnimal:
        cls = self.__class__
        return cls(name, self.birthday)

class Dog(Animal):
    def bark(self) -> None:
        print(f"{self.name} says woof!")

fido = Dog.newborn("Fido")
pluto = fido.twin("Pluto")
fido.bark()
pluto.bark()
```
There are a few things to note in this example:
- The type variable `TAnimal` is used to denote that *return values might be instances of subclasses of `Animal`*.
- We specify that `Animal` is an upper bound for `TAnimal`. Specifying bound means that `TAnimal` will only be `Animal` or one of its subclasses. This is needed to properly restrict the types that are allowed.
- The `typing.Type[]` construct is the typing equivalent of `type()`. You need it to note that the class method expects a class and returns an instance of that class.

### Annotating *args and **kwargs
In the object oriented version of the game, we added the option to name the players on the command line. This is done by listing player names after the name of the program:
```bash
$ python game.py GeirArne Dan Joanna
Dan: ♢A   Joanna: ♡9   P1: ♣A   GeirArne: ♣2
Dan: ♡A   Joanna: ♡6   P1: ♠4   GeirArne: ♢8
Dan: ♢K   Joanna: ♢Q   P1: ♣K   GeirArne: ♠5
Dan: ♡2   Joanna: ♡J   P1: ♠7   GeirArne: ♡K
Dan: ♢10  Joanna: ♣3   P1: ♢4   GeirArne: ♠8
Dan: ♣6   Joanna: ♡Q   P1: ♣Q   GeirArne: ♢J
Dan: ♢2   Joanna: ♡4   P1: ♣8   GeirArne: ♡7
Dan: ♡10  Joanna: ♢3   P1: ♡3   GeirArne: ♠2
Dan: ♠K   Joanna: ♣5   P1: ♣7   GeirArne: ♠J
Dan: ♠6   Joanna: ♢9   P1: ♣J   GeirArne: ♣10
Dan: ♠3   Joanna: ♡5   P1: ♣9   GeirArne: ♠Q
Dan: ♠A   Joanna: ♠9   P1: ♠10  GeirArne: ♡8
Dan: ♢6   Joanna: ♢5   P1: ♢7   GeirArne: ♣4
```
This is implemented by unpacking and passing in `sys.argv` to `Game()` when it’s instantiated. The `.__init__()` method uses `*names` to pack the given names into a tuple.

Regarding type annotations: even though names will be a tuple of strings, you should only annotate the type of each name. In other words, you should use str and not `Tuple[str]`:
```python
class Game:
    def __init__(self, *names: str) -> None:
        """Set up the deck and deal cards to 4 players"""
        deck = Deck.create(shuffle=True)
        self.names = (list(names) + "P1 P2 P3 P4".split())[:4]
        self.hands = {
            n: Player(n, h) for n, h in zip(self.names, deck.deal(4))
        }
```
Similarly, if you have a function or method accepting `**kwargs`, then you should only annotate the type of each possible keyword argument.

### Callables
Functions are first-class objects in Python. This means that you can use functions as arguments to other functions. That also means that you need to be able to add type hints representing functions.

Functions, as well as lambdas, methods and classes, are represented by `typing.Callable`. The types of the arguments and the return value are usually also represented. For instance, `Callable[[A1, A2, A3], Rt]` represents a function with three arguments with types `A1`, `A2`, and `A3`, respectively. The return type of the function is `Rt`.

In the following example, the function `do_twice()` calls a given function twice and prints the return values:
```python
# do_twice.py
from typing import Callable

def do_twice(func: Callable[[str], str], argument: str) -> None:
    print(func(argument))
    print(func(argument))

def create_greeting(name: str) -> str:
    return f"Hello {name}"

do_twice(create_greeting, "Jekyll")
```
Note the annotation of the `func` argument to `do_twice()` on line 5. It says that `func` should be a callable with one string argument, that also returns a string. One example of such a callable is `create_greeting()` defined on line 9.

## The typing Module
Before to conclude this lecture, let us introduce a survey of all the typing possibilities with the `typing` module. 

The `typing` module was introduced in the standard library to add many datatype for static type checking to Python 3.5 as well as older versions. 

It defines the fundamental building blocks for constructing types (e.g. `Any`), types representing generic variants of builtin collections (e.g. `List`), types representing generic collection ABCs (e.g. `Sequence`), and a small collection of convenience definitions.

Note that special type constructs, such as `Any`, `Union`, and type variables defined using `TypeVar` are only supported in the type annotation context, and `Generic` may only be used as a base class. All of these (except for unparameterized generics) will raise `TypeError` if appear in `isinstance` or `issubclass`.

### Fundamental building blocks:
- `Any`, used as `def get(key: str) -> Any: ...`.
- `Union`, used as `Union[Type1, Type2, Type3]`.
- `Callable`, used as `Callable[[Arg1Type, Arg2Type], ReturnType]`.
- `Tuple`, used by listing the element types, for example `Tuple[int, int, str]`. The empty tuple can be typed as `Tuple[()]`. Arbitrary-length homogeneous tuples can be expressed using one type and ellipsis, for example `Tuple[int, ...]`. (The `...` here are part of the syntax, a literal ellipsis.)
- `TypeVar`, used as `X = TypeVar('X', Type1, Type2, Type3)` or simply `Y = TypeVar('Y')` (see below for more details).
- `Generic`, used to create user-defined generic classes.
- `Type`, used to annotate class objects.

`Generic` variants of builtin collections:
- `Dict`, used as `Dict[key_type, value_type]`.
- `DefaultDict`, used as `DefaultDict[key_type, value_type]`, a generic variant of `collections.defaultdict`.
- `List`, used as `List[element_type]`.
- `Set`, used as `Set[element_type]`. See remark for `AbstractSet` below.
- `FrozenSet`, used as `FrozenSet[element_type]`.

Note: `Dict`, `DefaultDict`, `List`, `Set` and `FrozenSet` are mainly useful for annotating return values. For arguments, prefer the abstract collection types defined below, e.g. `Mapping`, `Sequence` or `AbstractSet`.

### `Generic` variants of container ABCs (and a few non-containers):
- `Awaitable`.
- `AsyncIterable`.
- `AsyncIterator`.
- `ByteString`.
- `Callable` (see above, listed here for completeness).
- `Collection`.
- `Container`.
- `ContextManager`.
- `Coroutine`.
- `Generator`, used as `Generator[yield_type, send_type, return_type]`. This represents the return value of generator functions. It is a subtype of `Iterable` and it has additional type variables for the type accepted by the `send()` method (it is contravariant in this variable – a generator that accepts sending it `Employee` instance is valid in a context where a generator is required that accepts sending it `Manager` instances) and the return type of the generator.
- `Hashable` (not generic, but present for completeness).
- `ItemsView`.
- `Iterable`.
- `Iterator`.
- `KeysView`.
- `Mapping`.
- `MappingView`.
- `MutableMapping`.
- `MutableSequence`.
- `MutableSet`.
- `Sequence`.
- `Set`, renamed to `AbstractSet`. This name change was required because `Set` in the typing module means `set()` with generics.
- `Sized` (not generic, but present for completeness).
- `ValuesView`.

### Single special methods
A few one-off types are defined that test for single special methods (similar to `Hashable` or `Sized`):
- `Reversible`, to test for `__reversed__`.
- `SupportsAbs`, to test for `__abs__`.
- `SupportsComplex`, to test for `__complex__`.
- `SupportsFloat`, to test for `__float__`.
- `SupportsInt`, to test for `__int__`.
- `SupportsRound`, to test for `__round__`.
- `SupportsBytes`, to test for `__bytes__`.

### Convenience definitions:
- `Optional`, defined by `Optional[t] == Union[t, None]`.
- `Text`, a simple alias for `str` in Python 3, for unicode in Python 2.
- `AnyStr`, defined as `TypeVar('AnyStr', Text, bytes)`.
- `NamedTuple`, used as `NamedTuple(type_name, [(field_name, field_type), ...])` and equivalent to collections.`namedtuple(type_name, [field_name, ...])`. This is useful to declare the types of the fields of a named tuple type.
- `NewType`, used to create unique types with little runtime overhead `UserId = NewType('UserId', int)`.
- `cast()`, described below.
- `no_type_check`, a decorator to disable type checking per class or function (see below).
- `no_type_check_decorator`, a decorator to create your own decorators with the same meaning as `@no_type_check` (see below).
- `type_check_only`, a decorator only available during type checking for use in stub files (see above); marks a class or function as unavailable during runtime.
- `overload`, described earlier.
- `get_type_hints()`, a utility function to retrieve the type hints from a function or method. Given a function or method object, it returns a dict with the same format as `__annotations__`, but evaluating forward references (which are given as string literals) as expressions in the context of the original function or method definition.
- `TYPE_CHECKING`, `False` at runtime but `True` to type checkers.

### I/O related types:
- `IO` (generic over `AnyStr`).
- `BinaryIO` (a simple subtype of `IO[bytes]`).
- `TextIO` (a simple subtype of `IO[str]`).

### Regular expressions and the `re` module
Types related to regular expressions and the re module:
- `Match` and `Pattern`, types of `re.match()` and `re.compile()` results (generic over `AnyStr`).

## Conclusion: Style guide for Python code
This lecture ends by a very important PEP: the [Style guide for Python code](https://peps.python.org/pep-0008/). This PEP recaps all the conventions for writing code in Python. 

It should be noticed here some interesting facts. 
- First, there is no strict rules, since Python has an huge ecosystem with many different API coming with contradictory naming conventions... 
- Second, nevertheless and if possible, it is encouraged to follow these conventions for functions, identifiers, classes...
- Third and last, the PEP ends with the typing considerations, that can be summarized as follows: typing is widely encouraged in new code, especially for interface or module. 

So the final word of Lecture 2 could be the following: **Enjoy typing!**

## Checkpoint, and where this goes next

You can now type generic code and object-oriented code: `TypeVar` in its three flavours,
`Optional` for the `None` sentinel, classes as types, forward references, `self` / `cls`
with a bounded type variable, `*args` / `**kwargs`, and `Callable`.

That is **Labwork 2, exercises 3 and 4** — the generic linked list, which is `TypeVar`
plus `Optional` almost line for line, and the re-annotated vehicle hierarchy. From this
unit onwards, every labwork is expected to pass:

```bash
mypy --strict .
```

**Into the rest of the week.** `Callable` is how Lecture 3 annotates a mock; `Optional`
without a guard is one of the errors `--strict` catches in Labwork 4; and the project's
`Dual` and `Tensor` classes are typed exactly as `Card` and `Deck` were here.

**Into Week 2.** Duck typing, met informally in 2a, becomes explicit: Lecture 4 of this
week writes interfaces as `ABC` + `@abstractmethod`, and all of Week 2's `optlab` is built
on them. `typing.Protocol` — structural typing, duck typing that the checker can
verify — is the other half of that story; you do not need it this week, but now you know
the word.
